In [91]:
import re
import tiktoken
from chromadb import Client
from langchain_groq import ChatGroq
from chromadb.config import Settings
from typing import TypedDict, List, AnyStr
from langchain_core.messages import SystemMessage
from docling.document_converter import DocumentConverter
from sentence_transformers import SentenceTransformer, CrossEncoder

In [112]:
class Query_Output(TypedDict):
    sections : List

In [141]:
query_llm = ChatGroq(model="llama-3.3-70b-versatile").with_structured_output(Query_Output)

In [151]:
SystemPrompt = """
You are a query refinement and section selection assistant for a Retrieval-Augmented Generation (RAG) system working on research papers.

Your tasks:
2. **Select relevant sections**: From the provided list of sections in the research paper, choose the ones that are most likely to contain information answering the query. Only select sections that are genuinely relevant — do NOT return all sections by default.

### Rules:
- If multiple sections could contain useful information, include all of them.
- If you are including a section, try including the subsections as well as they might have info similar to the main section.
- If you are uncertain, prioritize "Abstract", "Introduction", and "Discussion" as general fallback sections.

Be concise, precise, and always valid JSON.
                             
###Inputs:
                   
Query: {Query}
sections: 
{sections}                          
"""

In [ ]:
def get_heading(chunk):
    match = re.match(r"^#{1,6}\s+(.*)", chunk.strip().splitlines()[0])
    return match.group(1).strip() if match else "Unknown"

enc = tiktoken.get_encoding("cl100k_base")
embedder = SentenceTransformer("allenai/specter2_base")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

#embedder = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased", torch_dtype="auto")

2025-09-16 14:13:16,090 - INFO - Use pytorch device_name: cpu
2025-09-16 14:13:16,103 - INFO - Load pretrained SentenceTransformer: allenai/specter2_base
2025-09-16 14:13:17,727 - WARNING - No sentence-transformers model found with name allenai/specter2_base. Creating a new one with mean pooling.


In [155]:
path = r"C:\\Users\\Zigron\\Documents\\research papers\\LIME.pdf"
converter = DocumentConverter()

In [156]:
doc = converter.convert(path).document

2025-09-16 16:16:19,504 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-09-16 16:16:19,667 - INFO - Going to convert document batch...
2025-09-16 16:16:19,680 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 409740a308d273b3f090712d250cd783
2025-09-16 16:16:19,724 - INFO - Accelerator device: 'cpu'
2025-09-16 16:16:28,735 - INFO - Accelerator device: 'cpu'
2025-09-16 16:16:31,623 - INFO - Accelerator device: 'cpu'
2025-09-16 16:16:34,716 - INFO - Processing document LIME.pdf
c:\Users\Zigron\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Zigron\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.

In [157]:
md = doc.export_to_markdown()

In [158]:
import re
import tiktoken

# ---- Tokenizer ----
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def split_long_chunk(text, section, position, max_tokens=800, overlap=100):
    tokens = enc.encode(text)
    chunks = []
    start = 0
    sub_pos = 0

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = enc.decode(chunk_tokens)

        chunks.append({
            "text": chunk_text,
            "metadata": {
                "section": section,
                "position": f"{position}.{sub_pos}"  # keep sub-index
            }
        })

        start += max_tokens - overlap
        sub_pos += 1

    return chunks

# ---- Chunking Pipeline ----
def chunk_markdown(md: str, max_tokens=800, overlap=100):
    # Find all heading positions
    positions = [m.start() for m in re.finditer(r"^#{1,6}\s", md, re.MULTILINE)]
    positions = positions[1:]

    chunks = []
    for i in range(len(positions)-1):
        chunks.append(md[positions[i]:positions[i+1]])
    chunks.append(md[positions[-1]:])  # last section

    final_chunks = []
    for idx, chunk in enumerate(chunks):
        # Extract section heading
        first_line = chunk.strip().splitlines()[0]
        heading_match = re.match(r"^#{1,6}\s+(.*)", first_line)
        section = heading_match.group(1).strip() if heading_match else "Unknown"

        if count_tokens(chunk) > max_tokens:
            final_chunks.extend(split_long_chunk(chunk, section, idx, max_tokens, overlap))
        else:
            final_chunks.append({
                "text": chunk,
                "metadata": {
                    "section": section,
                    "position": str(idx)
                }
            })

    return final_chunks


In [159]:
chunks = chunk_markdown(md)

In [160]:
sections = set([chunk['metadata']['section'] for chunk in chunks])

In [161]:
for ch in chunks:
    ch["embeddings"] = embedder.encode(ch["text"])

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.87it/s]


In [162]:
client = Client(Settings(persist_directory="./chroma_store"))
collection = client.get_or_create_collection("papers")

In [163]:
collection.add(
    ids=[c["metadata"]["position"] for c in chunks],
    documents=[c["text"] for c in chunks],
    embeddings=[c["embeddings"] for c in chunks],
    metadatas=[c["metadata"] for c in chunks]
)

In [164]:
llm_response = query_llm.invoke(SystemPrompt.format(Query="Summarize the findings of this paper.",sections=sections))

2025-09-16 16:20:05,136 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [165]:
results = []
for section in llm_response['sections']:
    results.append(collection.query(
    query_embeddings=embedder.encode("Summarize the findings of this paper."),
    n_results=10,
    where = {"section": section}
))

Batches: 100%|██████████| 1/1 [00:00<00:00, 12.05it/s]


In [166]:
for res in results:
    print(res['documents'])

[[]]
[[" instances with explanations to address the 'trusting the model' problem, via submodular optimization.\n- Comprehensive evaluation with simulated and human subjects, where we measure the impact of explanations on trust and associated tasks. In our experiments, non-experts using LIME are able to pick which classifier from a pair generalizes better in the real world. Further, they are able to greatly improve an untrustworthy classifier trained on 20 newsgroups, by doing feature engineering using LIME. We also show how understanding the predictions of a neural network on images helps practitioners know when and why they should not trust a model.\n\n", "## 1. INTRODUCTION\n\nMachine learning is at the core of many recent advances in science and technology. Unfortunately, the important role of humans is an oft-overlooked aspect in the field. Whether humans are directly using machine learning classifiers as tools, or are deploying models within other products, a vital concern remains

In [167]:
llm_response['sections']

['ABSTRACT', '1. INTRODUCTION', '8. CONCLUSION AND FUTURE WORK']

In [154]:
sections

{'1. Introduction',
 '2. Related Work',
 '2.1. Vision-language Pre-training',
 '2.2. Knowledge Distillation',
 '2.3. Data Augmentation',
 '3. Method',
 '3.1. Model Architecture',
 '3.2. Pre-training Objectives',
 '3.3. CapFilt',
 '4. Experiments and Discussions',
 '4.1. Pre-training Details',
 '4.2. Effect of CapFilt',
 '4.3. Diversity is Key for Synthetic Captions',
 '4.4. Parameter Sharing and Decoupling',
 '5. Comparison with State-of-the-arts',
 '5.1. Image-Text Retrieval',
 '5.2. Image Captioning',
 '5.3. Visual Question Answering (VQA)',
 '5.4. Natural Language Visual Reasoning (NLVR 2 )',
 '5.5. Visual Dialog (VisDial)',
 '5.6. Zero-shot Transfer to Video-Language Tasks',
 '6. Additional Ablation Study',
 '7. Conclusion',
 'A. Downstream Task Details',
 'Abstract',
 'B. Additional Examples of Synthetic Captions',
 'C. Pre-training Dataset Details',
 'References'}